In [1]:
import pandas as pd

ftse = pd.read_csv("ftse100_close_2015_2026.csv", parse_dates=["date"], index_col="date")["close"]
spx = pd.read_csv("sp500_close_2015_2026.csv", parse_dates=["date"], index_col="date")["close"]
print(ftse.shape, spx.shape)

(2956,) (2942,)


In [2]:
import pmdarima as pm

auto_modelo = pm.auto_arima(ftse, start_p=0, start_q=0, max_p=5, max_q=5, d=None,
                             seasonal=False, stepwise=True, trace=True,
                             suppress_warnings=True, information_criterion="aic")
print(auto_modelo.summary())

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=33206.174, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=33208.170, Time=0.06 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=33208.178, Time=0.03 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=33205.462, Time=0.01 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=33210.178, Time=0.04 sec

Best model:  ARIMA(0,1,0)(0,0,0)[0]          
Total fit time: 0.176 seconds
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 2956
Model:               SARIMAX(0, 1, 0)   Log Likelihood              -16601.731
Date:                Wed, 16 Sep 2026   AIC                          33205.462
Time:                        18:54:47   BIC                          33211.453
Sample:                             0   HQIC                         33207.619
                               - 2956                                  

In [3]:
print("Seleccion manual (leccion 08.02, grid p=1..5 con d=1,q=1 fijo): ARIMA(1,1,1), AIC =", 33207.72)
print("auto_arima (busqueda libre de p,d,q)                       :", auto_modelo.order,
      ", AIC =", round(auto_modelo.aic(), 2))

Seleccion manual (leccion 08.02, grid p=1..5 con d=1,q=1 fijo): ARIMA(1,1,1), AIC = 33207.72
auto_arima (busqueda libre de p,d,q)                       : (0, 1, 0) , AIC = 33205.46


In [4]:
spx_ret = spx.pct_change().dropna() * 100
datos = pd.concat([ftse, spx_ret], axis=1, join="inner")
datos.columns = ["ftse_close", "spx_ret"]

auto_modelo_exog = pm.auto_arima(datos["ftse_close"], X=datos[["spx_ret"]],
                                  start_p=0, start_q=0, max_p=5, max_q=5, d=None,
                                  seasonal=False, stepwise=True, trace=True,
                                  suppress_warnings=True, information_criterion="aic")
print(auto_modelo_exog.summary())

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=32367.663, Time=0.01 sec


 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=32368.081, Time=0.03 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=32368.043, Time=0.05 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=32367.091, Time=0.07 sec


 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=32364.880, Time=0.21 sec


 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=32366.670, Time=0.30 sec


 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=32366.667, Time=0.28 sec
 ARIMA(0,1,2)(0,0,0)[0] intercept   : AIC=32369.544, Time=0.05 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=32369.587, Time=0.05 sec


...

 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=32365.833, Time=0.67 sec


 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=32368.664, Time=0.64 sec


 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=32368.720, Time=0.49 sec


 ARIMA(3,1,3)(0,0,0)[0] intercept   : AIC=32372.156, Time=1.02 sec


 ARIMA(2,1,2)(0,0,0)[0]             : AIC=32365.246, Time=0.30 sec

Best model:  ARIMA(2,1,2)(0,0,0)[0] intercept
Total fit time: 5.501 seconds
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                 2887
Model:               SARIMAX(2, 1, 2)   Log Likelihood              -16175.340
Date:                Wed, 16 Sep 2026   AIC                          32364.679
Time:                        18:54:53   BIC                          32406.453
Sample:                             0   HQIC                         32379.735
                               - 2887                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.0866      0.074      1.174      0.241      -0.058

In [5]:
print("Comparacion final de AIC (mismos 2887 dias, FTSE + S&P 500 como exogena):")
print("  ARIMAX(1,1,1) manual, seleccion a mano (leccion 08.02):", 32364.28)
print("  auto_arima con exogena                                :", round(auto_modelo_exog.aic(), 2))
print("  orden elegido por auto_arima                          :", auto_modelo_exog.order)

Comparacion final de AIC (mismos 2887 dias, FTSE + S&P 500 como exogena):
  ARIMAX(1,1,1) manual, seleccion a mano (leccion 08.02): 32364.28
  auto_arima con exogena                                : 32364.68
  orden elegido por auto_arima                          : (2, 1, 2)
